# 🛡️ VoiceShield v3: Automated Hardened Fine-Tuning & Multi-Condition Validation
### Defense Against Replay Attacks, Room Reverberation, Telephonic Distortion & Frontier AI Voices

This automated pipeline upgrades `ArmanSandhu1729/voiceshield-hindi-english-deepfake-v2` to **v3-hardened**.

### 🎯 What Makes v3 Resilient to Failures:
1. **Acoustic Room Impulse Simulation (Replay Defense)**: Simulates loudspeaker playback through air into physical microphones via RIR convolutions.
2. **Telephonic & Distortion Invariance**: Simulates narrowband (300Hz - 3,400Hz) telephone filtering and non-linear audio distortion.
3. **Ambient Room Noise Injection**: Exposes the model to varied room SNR levels so environmental noise is never confused with human voice.
4. **Frontier & Conversational AI Ingestion**: Drop-in support for Gemini Audio, F5-TTS, and Sarvam Bulbul clips.
5. **Automated End-to-End Validation**: Evaluates the hardened model on unseen clean, distorted, and reverberant test voices.

**Hardware Required**: Google Colab Free-Tier **T4 GPU** (~12-15 minutes).

## Step 1: Install Required AI & Audio Frameworks

In [ ]:
# Step 1: Install required packages (Run this first!)
!pip install -q --upgrade transformers accelerate datasets soundfile torchaudio evaluate edge-tts huggingface_hub librosa scipy
print('✅ Audio & ML dependencies installed successfully!')

## Step 2: Hugging Face Login
Get your **Write** token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) to push the final v3-hardened model.

In [ ]:
from huggingface_hub import login
login()

## Step 3: Automated Dataset Generation with Acoustic Channel & Distortion Augmentation
Streams 500 English Real + 500 Hindi Real (Google FLEURS) and synthesizes 500 English AI + 500 Hindi AI (Edge-TTS).
Applies **Acoustic Channel Simulation (Room Reverb, Telephonic Bandpass, Ambient Noise)** to build immunity against microphone re-recording and distortion.

In [ ]:
import os, glob, time, asyncio, random
import numpy as np, soundfile as sf, librosa, scipy.signal, edge_tts
from datasets import load_dataset

os.makedirs('dataset_v3/real', exist_ok=True)
os.makedirs('dataset_v3/fake', exist_ok=True)
os.makedirs('dataset_v3/custom_fake', exist_ok=True)

# =====================================================
# ACOUSTIC CHANNEL SIMULATION & AUGMENTATION OPERATORS
# =====================================================
def apply_room_reverb(audio, sr=16000, decay_sec=0.22):
    t = np.linspace(0, decay_sec, int(sr * decay_sec))
    rir = np.exp(-t * 14) * np.random.randn(len(t))
    rir[0] = 1.0
    rir /= np.sum(np.abs(rir))
    rev = scipy.signal.convolve(audio, rir, mode='same')
    return (0.75 * audio + 0.25 * rev).astype(np.float32)

def apply_distortion_and_bandpass(audio, sr=16000):
    sos = scipy.signal.butter(4, [300, 3400], btype='bandpass', fs=sr, output='sos')
    filtered = scipy.signal.sosfilt(sos, audio)
    dist = np.tanh(filtered * 1.5)
    max_val = np.max(np.abs(dist)) + 1e-6
    return (dist / max_val).astype(np.float32)

def apply_ambient_noise(audio, snr_db=22):
    noise = np.random.randn(len(audio))
    sig_power = np.mean(audio ** 2)
    noise_power = np.mean(noise ** 2)
    factor = np.sqrt(sig_power / (10 ** (snr_db / 10) * noise_power + 1e-9))
    return (audio + factor * noise).astype(np.float32)

def augment_audio(audio, sr=16000):
    # 25% Room Reverb (Replay simulation)
    # 25% Distortion & Phone Bandpass
    # 25% Ambient Room Noise
    # 25% Pristine Clean Digital
    choice = random.random()
    if choice < 0.25:
        return apply_room_reverb(audio, sr)
    elif choice < 0.50:
        return apply_distortion_and_bandpass(audio, sr)
    elif choice < 0.75:
        return apply_ambient_noise(audio)
    return audio

# =====================================================
# 1. REAL HUMAN SPEECH (500 English + 500 Hindi)
# =====================================================
print('📥 1. Streaming Real English Speech (Google FLEURS)...')
count_en = len(glob.glob('dataset_v3/real/en_real_*.wav'))
if count_en < 500:
    for split in ['train', 'validation']:
        if count_en >= 500: break
        ds = load_dataset('google/fleurs', 'en_us', split=split, streaming=True)
        for item in ds:
            if count_en >= 500: break
            y = item['audio']['array']
            sr = item['audio']['sampling_rate']
            if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
            if len(y) >= 16000 * 2.5:
                clip = augment_audio(y[:16000 * 4])
                sf.write(f'dataset_v3/real/en_real_{count_en}.wav', clip, 16000)
                count_en += 1
print(f'   ✅ {count_en} Real English clips ingested & augmented.')

print('📥 2. Streaming Real Hindi Speech (Google FLEURS)...')
count_hi = len(glob.glob('dataset_v3/real/hi_real_*.wav'))
if count_hi < 500:
    for split in ['train', 'validation']:
        if count_hi >= 500: break
        ds = load_dataset('google/fleurs', 'hi_in', split=split, streaming=True)
        for item in ds:
            if count_hi >= 500: break
            y = item['audio']['array']
            sr = item['audio']['sampling_rate']
            if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
            if len(y) >= 16000 * 2.5:
                clip = augment_audio(y[:16000 * 4])
                sf.write(f'dataset_v3/real/hi_real_{count_hi}.wav', clip, 16000)
                count_hi += 1
print(f'   ✅ {count_hi} Real Hindi clips ingested & augmented.')

# =====================================================
# 2. AI SYNTHETIC VOICES (500 English + 500 Hindi)
# =====================================================
print('\n🤖 3. Synthesizing AI Voices with Acoustic Simulation...')
en_voices = ['en-US-GuyNeural', 'en-US-JennyNeural', 'en-US-AriaNeural', 'en-GB-RyanNeural', 'en-IN-NeerjaNeural']
hi_voices = ['hi-IN-MadhurNeural', 'hi-IN-SwaraNeural']
pitch_vars = ['+0Hz', '+15Hz', '-15Hz']
rate_vars = ['+0%', '+5%', '-5%']

en_prompts = [
    'This is a high-security automated transaction notice from your banking institution.',
    'Please be informed that identity verification protocols require your immediate attention.',
    'Recent advances in voice synthesis enable authentic simulation of human conversational speech.',
    'Confirm your confidential six-digit passcode to proceed with this international wire transfer.',
    'All our support agents are currently assisting other callers. Please stay on the line.'
]
hi_prompts = [
    'नमस्ते, यह आपके बैंक की सुरक्षा शाखा से एक अत्यंत आवश्यक स्वचालित कॉल है।',
    'साइबर सुरक्षा नियमों का पालन करना आपके वित्तीय डेटा को इंटरनेट पर सुरक्षित रखने के लिए आवश्यक है।',
    'कृपया किसी भी व्यक्ति के साथ अपना गोपनीय ओटीपी या बैंक पासवर्ड कभी साझा न करें।',
    'आपके पंजीकृत पते पर नया पार्सल भेज दिया गया है, जो अगले दो कार्य दिवसों में मिल जाएगा।',
    'आर्टिफिशियल इंटेलिजेंस द्वारा तैयार की गई आवाज़ें बहुत स्वाभाविक और मानवीय लगती हैं।'
]

sem = asyncio.Semaphore(15)
async def gen_hardened(text, voice, out_name, rate='+0%', pitch='+0Hz'):
    async with sem:
        mp3 = f'{out_name}.mp3'
        wav = f'{out_name}.wav'
        if os.path.exists(wav): return
        for _ in range(2):
            try:
                comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
                await comm.save(mp3)
                y, _ = librosa.load(mp3, sr=16000)
                # Apply acoustic channel augmentation to AI audio (reverb, distortion, noise)
                y_aug = augment_audio(y[:16000 * 4])
                sf.write(wav, y_aug, 16000)
                if os.path.exists(mp3): os.remove(mp3)
                break
            except Exception:
                await asyncio.sleep(1)

async def run_ai():
    tasks = []
    for i in range(500):
        v = en_voices[i % len(en_voices)]
        p = en_prompts[i % len(en_prompts)]
        tasks.append(gen_hardened(p, v, f'dataset_v3/fake/en_ai_{i}'))
    for i in range(500):
        v = hi_voices[i % len(hi_voices)]
        p = hi_prompts[i % len(hi_prompts)]
        r = rate_vars[i % len(rate_vars)]
        pi = pitch_vars[i % len(pitch_vars)]
        tasks.append(gen_hardened(p, v, f'dataset_v3/fake/hi_ai_{i}', rate=r, pitch=pi))
    await asyncio.gather(*tasks)

await run_ai()
print('   ✅ 500 English AI + 500 Hindi AI clips generated & augmented.')

# Include custom frontier AI samples (Gemini, F5-TTS, Bulbul)
custom = glob.glob('dataset_v3/custom_fake/*.wav') + glob.glob('dataset_v3/custom_fake/*.mp3')
for idx, cpath in enumerate(custom):
    y, _ = librosa.load(cpath, sr=16000)
    sf.write(f'dataset_v3/fake/custom_{idx}.wav', y[:16000 * 4], 16000)
if custom: print(f'   ✅ Ingested {len(custom)} frontier custom AI clips.')

print('\n' + '=' * 60)
print('📊 HARDENED v3 BALANCED DATASET READY')
print(f'Total Real Human: {len(glob.glob("dataset_v3/real/*.wav"))}')
print(f'Total AI Voices:  {len(glob.glob("dataset_v3/fake/*.wav"))}')
print('=' * 60)


## Step 4: Load v2 Fine-Tuned Checkpoint for v3 Hardening
Instead of training from scratch, we start from `ArmanSandhu1729/voiceshield-hindi-english-deepfake-v2`.
This retains all existing bilingual weights while adapting specifically to overcome room reverberation, distortion, and replay attacks.

In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

base_checkpoint = 'ArmanSandhu1729/voiceshield-hindi-english-deepfake-v2'
print(f'🧠 Loading checkpoint: {base_checkpoint}...')

feature_extractor = AutoFeatureExtractor.from_pretrained(base_checkpoint)
model = AutoModelForAudioClassification.from_pretrained(
    base_checkpoint,
    num_labels=2,
    id2label={0: 'real', 1: 'fake'},
    label2id={'real': 0, 'fake': 1}
)

# Freeze lower CNN encoder to preserve core acoustic representations
model.freeze_feature_encoder()
print('✅ Checkpoint loaded with frozen convolutional feature encoder!')


## Step 5: Dataset Split & DataLoaders (85% Train / 15% Validation)

In [ ]:
import random
from torch.utils.data import Dataset

real_f = glob.glob('dataset_v3/real/*.wav')
fake_f = glob.glob('dataset_v3/fake/*.wav')
all_samples = [(f, 0) for f in real_f] + [(f, 1) for f in fake_f]
random.seed(42)
random.shuffle(all_samples)

split_pt = int(len(all_samples) * 0.85)
train_samples = all_samples[:split_pt]
val_samples = all_samples[split_pt:]
print(f'Train clips: {len(train_samples)} | Validation clips: {len(val_samples)}')

class HardenedDeepfakeDataset(Dataset):
    def __init__(self, items, extractor, max_duration=4.0):
        self.items = items
        self.extractor = extractor
        self.max_dur = max_duration
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        speech, sr = sf.read(path)
        if len(speech.shape) > 1: speech = speech[:, 0]
        inputs = self.extractor(
            speech,
            sampling_rate=16000,
            max_length=int(16000 * self.max_dur),
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        out = {k: v.squeeze(0) for k, v in inputs.items()}
        out['labels'] = torch.tensor(label, dtype=torch.long)
        return out

train_ds = HardenedDeepfakeDataset(train_samples, feature_extractor)
val_ds = HardenedDeepfakeDataset(val_samples, feature_extractor)
print('✅ PyTorch DataLoaders configured for v3 hardening.')


## Step 6: Fine-Tune v3-Hardened Model on T4 GPU (5 Epochs)
Uses AdamW with learning rate $2 \times 10^{-5}$ and FP16 mixed precision.

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate

acc_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc_metric.compute(predictions=preds, references=labels)

train_args = TrainingArguments(
    output_dir='./voiceshield-v3-hardened',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=15,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

print('🚀 Starting v3 Hardened Fine-Tuning on T4 GPU...')
trainer.train()
print('🎉 v3 Hardening Completed Successfully!')


## Step 7: Push v3-Hardened Model to Hugging Face Hub

In [ ]:
v3_target_repo = 'ArmanSandhu1729/voiceshield-hindi-english-deepfake-v3-hardened'
print(f'🚀 Publishing hardened v3 model to: {v3_target_repo}...')

trainer.save_model('./voiceshield-v3-hardened-final')
feature_extractor.save_pretrained('./voiceshield-v3-hardened-final')

model.push_to_hub(v3_target_repo)
feature_extractor.push_to_hub(v3_target_repo)

print(f'\n🏆 SUCCESS! Your hardened v3 model is now live at:')
print(f'👉 https://huggingface.co/{v3_target_repo}')


## Step 8: Automated Validation on Brand New Challenging Voices (Clean, Distorted & Replay)

In [ ]:
# =====================================================
# RIGOROUS MULTI-CONDITION BENCHMARK
# Tests on: Clean AI, Distorted AI, Re-recorded AI, Clean Human, Distorted Human
# =====================================================
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

print('⚡ Running Automated Multi-Condition Validation on Unseen Test Voices...')

# Build unseen test pool from official test splits
os.makedirs('val_v3/real_clean', exist_ok=True)
os.makedirs('val_v3/real_distorted', exist_ok=True)
os.makedirs('val_v3/fake_clean', exist_ok=True)
os.makedirs('val_v3/fake_reverb', exist_ok=True)
os.makedirs('val_v3/fake_distorted', exist_ok=True)

# Stream 50 fresh real test clips
ds_test = load_dataset('google/fleurs', 'hi_in', split='test', streaming=True)
test_cnt = 0
for item in ds_test:
    if test_cnt >= 50: break
    y = item['audio']['array']
    sr = item['audio']['sampling_rate']
    if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
    if len(y) >= 16000 * 3:
        # Save clean
        sf.write(f'val_v3/real_clean/human_{test_cnt}.wav', y[:16000 * 4], 16000)
        # Save distorted
        sf.write(f'val_v3/real_distorted/human_dist_{test_cnt}.wav', apply_distortion_and_bandpass(y[:16000 * 4]), 16000)
        test_cnt += 1

# Synthesize fresh AI test clips with varied perturbations
async def gen_test_pool():
    for i in range(50):
        comm = edge_tts.Communicate('यह सुरक्षा जांच के लिए एक नया परीक्षण संदेश है।', 'hi-IN-SwaraNeural', pitch='+10Hz')
        await comm.save(f'val_v3/temp_{i}.mp3')
        y, _ = librosa.load(f'val_v3/temp_{i}.mp3', sr=16000)
        sf.write(f'val_v3/fake_clean/ai_clean_{i}.wav', y[:16000 * 4], 16000)
        sf.write(f'val_v3/fake_reverb/ai_reverb_{i}.wav', apply_room_reverb(y[:16000 * 4]), 16000)
        sf.write(f'val_v3/fake_distorted/ai_dist_{i}.wav', apply_distortion_and_bandpass(y[:16000 * 4]), 16000)
        if os.path.exists(f'val_v3/temp_{i}.mp3'): os.remove(f'val_v3/temp_{i}.mp3')

await gen_test_pool()

test_sets = [
    ('Real Clean Human', glob.glob('val_v3/real_clean/*.wav'), 'real'),
    ('Real Distorted Human', glob.glob('val_v3/real_distorted/*.wav'), 'real'),
    ('AI Clean Voices', glob.glob('val_v3/fake_clean/*.wav'), 'fake'),
    ('AI Room Reverb / Replay', glob.glob('val_v3/fake_reverb/*.wav'), 'fake'),
    ('AI Distorted Voices', glob.glob('val_v3/fake_distorted/*.wav'), 'fake')
]

report_rows = []
print('\n' + '=' * 75)
print('📊 VOICESHIELD v3 MULTI-CONDITION VALIDATION SCORECARD')
print('=' * 75)

for category, files, expected in test_sets:
    correct = 0
    latencies = []
    for f in files:
        speech, _ = sf.read(f)
        t0 = time.time()
        inputs = feature_extractor(speech, sampling_rate=16000, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**inputs).logits.squeeze()
            probs = torch.softmax(logits / 2.4, dim=-1).cpu().tolist()
        latencies.append((time.time() - t0) * 1000)
        pred = 'fake' if probs[1] >= 0.50 else 'real'
        if pred == expected: correct += 1
        report_rows.append({
            'file': os.path.basename(f),
            'category': category,
            'expected': expected,
            'prediction': pred,
            'fake_prob': round(probs[1], 4)
        })
    acc = (correct / len(files)) * 100 if files else 0
    print(f'🔹 {category:<26}: {acc:>6.2f}% Accuracy ({correct}/{len(files)}) | Latency: {np.mean(latencies):.1f}ms')

print('=' * 75)
pd.DataFrame(report_rows).to_csv('voiceshield_v3_validation_report.csv', index=False)
print('📁 Exported full multi-condition validation report to: voiceshield_v3_validation_report.csv')
